In [6]:
# Model classes

import torch
import torch.nn as nn
import torchtune
from torchtune.models import llama3_2
from transformers import AutoModelForCausalLM

def _multinomial_sample_one_no_sync(probs):  # Does multinomial sampling without a cuda synchronization
    q = torch.empty_like(probs).exponential_(1)
    return torch.argmax(probs / q, dim=-1, keepdim=True).to(dtype=torch.int)

def sample_topk(logits: torch.Tensor, topk: int, temperature: float):
    logits = logits / temperature

    filter_value: float = -float("Inf")
    indices_to_remove = logits < torch.topk(logits, topk)[0][..., -1, None]
    scores_processed = logits.masked_fill(indices_to_remove, filter_value)
    scores_processed = torch.nn.functional.log_softmax(scores_processed, dim=-1)
    probs = torch.nn.functional.softmax(scores_processed, dim=-1)

    sample_token = _multinomial_sample_one_no_sync(probs)
    return sample_token

def qwen2_1500M(device="cuda"):
    # https://github.com/linkedin/Liger-Kernel?tab=readme-ov-file#supercharge-your-model-with-liger-kernel
    # Modify AutoModelForCausalLM classes with fused kernels
    from liger_kernel.transformers import apply_liger_kernel_to_qwen2
    apply_liger_kernel_to_qwen2()

    # Qwen 2.5 1.5B Instruct with extra 2048+10 tokens added to vocab
    return AutoModelForCausalLM.from_pretrained(
        "/workspace/tmp/qwen-1.5b-instruct-expanded-2048",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map=device
    )

def llama3_2_100M() -> torchtune.modules.transformer.TransformerDecoder:
    return llama3_2.llama3_2(
        vocab_size=128_256,
        num_layers=4,
        num_heads=8,
        num_kv_heads=2,
        embed_dim=1024,
        max_seq_len=2048,
        intermediate_dim=8192,
        attn_dropout=0.0,
        norm_eps=1e-5,
        rope_base=500_000,
        scale_factor=32,
    )

def _prepare_transformer(model):
    embed_dim = model.tok_embeddings.embedding_dim
    # No embedding lookup (pass in pre looked up embeddings)
    model.tok_embeddings = nn.Identity()
    # No lm_head
    model.output = nn.Identity()
    return model, embed_dim

# Code based on https://github.com/SesameAILabs/csm/blob/main/models.py
class CSMDepthDecoder(nn.Module):
    def __init__(
        self,
        csm_audio_vocab_size=2051,  # For some reason CSM vocab size is 2048+3
        csm_audio_num_codebooks=32,
        csm_backbone_dim=2048,
        # Rime depth decoder
        qwen_backbone_dim=1536,
        rime_num_codebooks=12
    ):
        super().__init__()

        self.csm_audio_vocab_size = csm_audio_vocab_size
        self.rime_num_codebooks = rime_num_codebooks

        # Just load all relevant weights in original shapes for now (can prune shapes later)
        self.decoder, decoder_dim = _prepare_transformer(llama3_2_100M())
        
        self.audio_embeddings = nn.Embedding(csm_audio_vocab_size * csm_audio_num_codebooks, csm_backbone_dim)
        
        self.projection = nn.Linear(csm_backbone_dim, decoder_dim, bias=False)
        self.codebook0_head = nn.Linear(csm_backbone_dim, csm_audio_vocab_size, bias=False)
        self.audio_head = nn.Parameter(torch.empty(csm_audio_num_codebooks - 1, decoder_dim, 2051))

        # Random init projection from Qwen hidden dim to CSM embedding dim
        self.qwenH_to_csmH = nn.Linear(qwen_backbone_dim, csm_backbone_dim, bias=False)
    
class DualDecoder(torch.nn.Module):

    def __init__(self, device="cuda"):
        super().__init__()
        self.device=device
        self.backbone = qwen2_1500M(device=device)
        self.decoder  = CSMDepthDecoder().to(device)

In [4]:
dd = DualDecoder()

Applied Liger kernels to Qwen2


In [3]:
cp=torch.load("/workspace/outputs/dd-with-csm-depth-weights/checkpoint_2487/dual-decoder.pt", map_location="cpu")

In [5]:
dd.load_state_dict(cp)

<All keys matched successfully>

In [9]:
from datasets import load_from_disk

ds = load_from_disk("/workspace/tmp/v4-crash-course/packed___700h-male_naramore_mikael/")

In [10]:
start_of_speech_index=ds[0]['input_ids'][0].index(151666)
prompt = ds[0]['input_ids'][0][ :(start_of_speech_index+1) ]

prompt = torch.LongTensor(prompt).unsqueeze(0)

prompt

tensor([[151644,   8948,    198,   3862,    572,    264,   4445,    315,  21162,
            304,    279,   3054,     11,    323,   1221,    279,  13428,  60174,
            448,  63526,     13,    362,   2421,   1251,  16009,    311,    862,
           7541,     11,    323,   1221,   5019,    770,   8110,    304,    264,
          11259,  24550,    367,    429,   1865,   8224,  69226,    448,  21770,
             13,   1260,   6966,   1495,    518,  56470,     11,   3498,     11,
            323,   5485,    264,  17811,   1401,    389,   1059,   3579,     11,
            323,  30056,   1549,    421,    566,   1030,  16925,  11882,    279,
           5492,    438,    264,   1616,    315,   5488,  46455,     13,  10696,
            566,   1186,    443,  70788,   3601,    311,   2746,     11,    566,
           3381,     13,   1988,   1221,    566,  34914,    432,   1007,     13,
           1260,   4172,  25882,  10960,   1091,    419,   7412,     13,   1260,
           4172,  25882,    

In [23]:
prompt.shape

torch.Size([1, 119])

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/workspace/tmp/qwen-1.5b-instruct-expanded-2048")

In [13]:
tokenizer.decode(prompt[0].tolist())

"<|im_start|>system\nThere was a moment of silence in the room, and then the crowd erupted with applause. A few people rose to their feet, and then everyone else followed in a standing ovation that made Sam blush with pride. He looked down at Indy, though, and saw a worried look on her face, and wondered again if he had somehow chosen the song as a way of saying goodbye. Maybe he subconsciously expected to die, he thought. But then he shook it off. He'd survived worse than this guy. He'd survived this too.<|im_end|>\n<|im_start|>assistant\n<custom_token_1>"

In [15]:
with torch.no_grad():
    backbone_outputs = dd.backbone.generate(
        input_ids=prompt.cuda(),
        attention_mask=torch.ones_like(prompt).cuda(),
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.5,
        top_p=1.0,
        repetition_penalty=1.1,
        num_return_sequences=1,
        eos_token_id=151_667,
        return_dict_in_generate=True,
        output_hidden_states=True
    )

In [43]:
# How many tokens generated? Not counting the final EndOfSpeech token
backbone_outputs.sequences.cpu()[:, prompt.shape[1]:-1].shape

torch.Size([1, 385])

In [83]:
last_layer_hidden_states = [ all_layers_hidden_states[-1] for all_layers_hidden_states in backbone_outputs.hidden_states[:-1] ]

In [109]:
self=dd
topk=50
temperature=0.9

generated_frames = []

for t in range(len(last_layer_hidden_states)):

    backbone_hidden_states=last_layer_hidden_states[t][:, -1, :]
    
    backbone_hidden_states = backbone_hidden_states.to(
        dtype=self.decoder.qwenH_to_csmH.weight.dtype # 
    )
    
    with torch.no_grad():
        csmH = self.decoder.qwenH_to_csmH(backbone_hidden_states)
        
        c0_logits = self.decoder.codebook0_head(csmH)
        c0_sample = sample_topk(c0_logits, topk, temperature)
        c0_embed = self.decoder.audio_embeddings(c0_sample)
    
        curr_h = torch.cat([ csmH.unsqueeze(1), c0_embed ], dim=1)
        curr_sample = c0_sample.clone()
    
        for i in range(1, self.decoder.rime_num_codebooks):
            decoder_h = self.decoder.decoder(self.decoder.projection(curr_h))
            
            ci_logits = torch.mm(decoder_h[:, -1, :], self.decoder.audio_head[i - 1])
            ci_sample = sample_topk(ci_logits, topk, temperature)
            ci_embed = self.decoder.audio_embeddings(ci_sample + i*2051)
    
            curr_h = ci_embed
            curr_sample = torch.cat([curr_sample, ci_sample], dim=1)
            
    generated_frames.append(curr_sample)

In [115]:
mimi_codes = torch.stack(generated_frames, dim=-1)

mimi_codes

tensor([[[ 799, 1952,   12,  ...,  385, 1164, 1400],
         [ 596,  919,  707,  ...,  201,  174,  481],
         [ 919,  378,  110,  ..., 1743,  788, 1445],
         ...,
         [1073, 1133, 1342,  ..., 1224, 1094, 1204],
         [1535, 1977, 1118,  ..., 1821, 1321,  913],
         [1507, 1163, 1041,  ...,  832,  890,  494]]], device='cuda:0',
       dtype=torch.int32)

In [117]:
from transformers import MimiModel

mimi_model = MimiModel.from_pretrained("kyutai/mimi").cuda()

In [119]:
from IPython.display import Audio

with torch.no_grad():
    reconstructed_audio = mimi_model.decode(mimi_codes).audio_values.cpu().numpy()[0, 0]

Audio(reconstructed_audio, rate=24_000)

In [120]:
import soundfile as sf

sf.write('/workspace/src/v4/csm-weights.wav', reconstructed_audio, 24_000)